In [0]:
from pyspark.sql import functions as F

catalog = "aircraft_data"
schema = "dataset"

silver_table = f"{catalog}.{schema}.silver_flight_data"

In [0]:
silver_df = spark.table(silver_table)

print(silver_df.count())

display(silver_df.limit(10))

In [0]:
#airline performance
gold_airline = (
    silver_df
    .groupBy("Reporting_Airline", "Airline_Name")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.sum("Cancelled").alias("CancelledFlights"),
        F.sum("Diverted").alias("DivertedFlights"),
        F.avg("ArrDelay").alias("AverageArrivalDelay"),
        F.avg("DepDelay").alias("AverageDepartureDelay")
    )
)

In [0]:
gold_airline.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_airline_performance")

In [0]:
#monthly performance
gold_monthly = (
    silver_df
    .groupBy("Year", "Month")
    .agg(
        F.count("*").alias("Flights"),
        F.sum("Cancelled").alias("CancelledFlights"),
        F.sum("Diverted").alias("DivertedFlights"),
        F.avg("ArrDelay").alias("AverageDelay")
    )
)

In [0]:
gold_monthly.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_monthly_summary")

In [0]:
#airport performance
gold_airport = (
    silver_df
    .groupBy("Origin", "OriginCityName")
    .agg(
        F.count("*").alias("Flights"),
        F.avg("DepDelay").alias("AverageDepartureDelay"),
        F.sum("Cancelled").alias("CancelledFlights")
    )
)

In [0]:
gold_airport.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_airport_performance")

In [0]:
#route performance
gold_route = (
    silver_df
    .groupBy("Route")
    .agg(
        F.count("*").alias("Flights"),
        F.avg("ArrDelay").alias("AverageArrivalDelay"),
        F.sum("Cancelled").alias("CancelledFlights")
    )
)

In [0]:
gold_route.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_route_performance")

In [0]:
#delay analysis
gold_delay = (
    silver_df
    .groupBy("DelayCategory")
    .agg(
        F.count("*").alias("Flights")
    )
)

In [0]:
gold_delay.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_delay")

In [0]:
#cancellation analysis
gold_cancel = (
    silver_df
    .groupBy("CancellationCode")
    .agg(
        F.count("*").alias("CancelledFlights")
    )
)

In [0]:
gold_cancel.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_cancellation")

In [0]:
#flight status
gold_status = (
    silver_df
    .groupBy("FlightPerformance")
    .agg(
        F.count("*").alias("Flights")
    )
)


In [0]:
gold_status.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_flight_status")


In [0]:
#kpi cards
gold_kpi = (
    silver_df
    .agg(
        F.count("*").alias("TotalFlights"),
        F.sum("Cancelled").alias("CancelledFlights"),
        F.sum("Diverted").alias("DivertedFlights"),
        F.avg("ArrDelay").alias("AverageArrivalDelay"),
        F.avg("DepDelay").alias("AverageDepartureDelay")
    )
)

In [0]:
gold_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aircraft_data.dataset.gold_kpi")
